# Reading the ball

**A traceable walkthrough of our frame-level possession model.**

More than a Hack 2027 · Barcelona vs Athletic Club. Prose and figures only: the code behind every number and clip is in `solution/`, and the appendix maps each number to its source. The configuration is frozen (D-029), so every number here belongs to the frozen build.

## 1. The problem: a ball that keeps disappearing

Watch the clip. Barcelona moves the ball in their own half, the players jog into shape, and the tracking loses the ball. Six seconds later it is still gone and the players have stopped. The answer strip under the pitch turns gray: the stretch is **dead**.

<img src="figs/fig_match_gap.gif" style="width:100%">
<div class="caption" style="font-size:0.9em; color:#555; margin:2px 0 14px 0;">Our match: live play, then the tracking loses the ball and the label turns D. Local render.</div>

Twenty seconds of our match, rendered from the submission's own tracking file. The gray band is a **ball-missing run**, frames with no ball position at all; the red band is the model's **D**. The ball is missing for 40.7% of graded frames, and almost all of that time is a stoppage. That fact is the spine of this notebook.

The task is one of **three labels** per graded frame: **H** Barcelona, **A** Athletic, **D** dead ball. A **possession** is a maximal unbroken run of H or A; D ends it. We answer at 25 Hz for 158,857 of 199,319 frames, and the sequence must be coherent.

<img src="figs/fig_three_labels.png" style="width:100%">
<div class="caption" style="font-size:0.9em; color:#555; margin:2px 0 14px 0;">The three labels, and the 25 answers per second they ask for. Synthetic.</div>

The obvious first rule is **who is nearest the ball**, the organizers' starter baseline, and with a 2.5 m radius it is a good start. But nearest is not **owner**: the owner is the team that last touched the ball and keeps it through a flight, a dribble or a scramble. In the schematic the defender is nearest, but the ball is still travelling from passer to receiver.

<img src="figs/fig_nearest_player.png" style="width:100%">
<div class="caption" style="font-size:0.9em; color:#555; margin:2px 0 14px 0;">Why the nearest player is not the owner: the ball is in flight, and the nearest player has not touched it. Synthetic.</div>

On clean practice data nearest catches .764 of visible owner changes; degraded to our noise, .482, and about 13.5% cannot be resolved from geometry at all. Hence two other clues: **missingness** and **team shape**.

## 2. The score: why one extra boundary is expensive

One true possession: Barcelona has the ball for ten seconds, then the whistle goes. The animation is synthetic and scored by the organizer's own metric, not a cartoon. It explains almost every design choice we made.

<img src="figs/fig_spellf1.gif" style="width:100%">
<div class="caption" style="font-size:0.9em; color:#555; margin:2px 0 14px 0;">The centerpiece: a true ten-second possession, a perfect answer, the same answer split, and a missed spell, scored by the organizer's metric. Synthetic.</div>

The score is **spellF1**. Possessions are matched one-to-one by **temporal IoU**, the shared frames divided by the union, and the F-score is averaged over thresholds 0.3–0.7. The right team is not enough if the interval is in the wrong place.

A perfect answer scores 1.00. Split it in two with a one-second D gap: .40, with each fragment at IoU 0.50 — inside the band, but only one can match and the other is a false positive. Split in three: .10, IoU 0.33, at the edge of the band. Split in four: .00, every fragment at IoU 0.25, below the 0.3 floor, so nothing matches. Miss it completely: also .00.

<img src="figs/fig_spellf1_curve.png" style="width:100%">
<div class="caption" style="font-size:0.9em; color:#555; margin:2px 0 14px 0;">The same experiment as a curve: score against the number of fragments. Synthetic.</div>

The asymmetry is the point. A missed possession costs one false negative; a spurious boundary costs a false positive *and* damages the host possession it cut, which must now compete with its own fragment. Hence our caution about hand-overs, and why a clue can raise frame accuracy and still lose score. Differences below about 0.04 are noise, and the public five minutes hold only 17 possessions, so we never choose a method on them.

## 3. The practice set: ten matches, made to look like ours

We have the true answers for five public minutes, and the rules forbid tuning on them: 17 possessions cannot choose a threshold, and tuning there picks the wrong answer even for a one-parameter baseline. Every choice is made on other professional matches with public answers, after we make their data look like ours.

The clip is one practice passage, clean and **degraded**. Clean, the ball stays visible through the stoppage, so a ball-based rule sees it for free. Degraded, the ball is dropped where our tracker drops it, the path is smoothed and the gap edges jittered: the stoppage is still there in the players, the ball is not.

<div style="display:flex;"><div style="flex:1; padding:0 4px;"><center><img src="figs/fig_practice_clean.gif" style="width:100%">
<div class="caption" style="font-size:0.9em; color:#555; margin:2px 0 14px 0;">The practice clip before degrading: the ball stays visible through the stoppage. Local render.</div></center></div><div style="flex:1; padding:0 4px;"><center><img src="figs/fig_practice_degraded.gif" style="width:100%">
<div class="caption" style="font-size:0.9em; color:#555; margin:2px 0 14px 0;">The same clip after degrading: the ball is missing where our tracker loses it. Local render.</div></center></div></div>

The practice set is **10 matches**: seven German-league and three Metrica. A second set of 20 SkillCorner matches is used only for who-has-the-ball questions, never pooled. The ten are frozen — same games, seeds, degrader and cells.

<img src="figs/fig_missingness.png" style="width:100%">
<div class="caption" style="font-size:0.9em; color:#555; margin:2px 0 14px 0;">Where the ball is missing: our match against the degraded practice set. Aggregates only.</div>

The set had to be repaired. Our match hides the ball for 40.7% of graded frames, the degraded practice set for 37.3%, close enough. The conditional level did not match: the old degrader dropped the ball at random, including in live play, so 42% of practice live frames had no ball against ~3% of ours. After the fix it is 4.4%, and 2.1% of long gaps are live against ≤ ~4% of ours. The old gap taught the model the opposite of the truth.

Two flaws, both fixed. The **resting ball** shook about twenty times too much, so ball motion separated dead from live far too well; fixing the sign took texture-only separability from perfect to .92, and .907 on twenty held-out SkillCorner games, still short of our .80 target — ball-motion clues are upper bounds. The second was **where the ball hides**: after the fix Baseline+ moved from .216 to .326 and the in-gap gain fell from +.082 to +.006. The set is also about .06 easier than the match — .3037 against .243 for the two-second baseline — so we trust the ranking, never the level.

## 4. The pipeline, one stage at a time

No single passage exercises all six stages, and the reason is structural: the gate needs a long ball-missing gap, grown D needs a visible resting ball, the decoder needs a gap the gate left live, and the sub-gap rule needs one it left alone. No window holds all four conditions, so the clips come from four passages — stages one to three share a window, and each later clip is the window where that stage changes a label.

**Stage 1 — nearest player, smoothed.** The organizers' two-second baseline: nearest player within a radius, majority-smoothed over fifty frames. The floor to beat: .3037.

<img src="figs/fig_stage_smooth50.gif" style="width:100%">
<div class="caption" style="font-size:0.9em; color:#555; margin:2px 0 14px 0;">Stage 1, the organizers' 2-second baseline: nearest player, majority-smoothed. Local render.</div>

**Stage 2 — hysteresis and merge.** A player within 2.5 m owns the ball, otherwise the last owner keeps it. It scores .3259, +.0222 [+.0058, +.0403] over the floor, but that lead does not survive a half-second shift, so read it as *about level to slightly ahead*. Too much hysteresis absorbs a turnover; too little emits flickers, which cost twice.

<img src="figs/fig_stage_baseline_plus.gif" style="width:100%">
<div class="caption" style="font-size:0.9em; color:#555; margin:2px 0 14px 0;">Stage 2, Baseline+: hysteresis, merge and a 2.5 m radius. Local render.</div>

**Stage 3 — the missing-ball gate.** Ball gone for a while and players still: the stretch becomes D. The largest gain in the stack, +.0728 [+.0649, +.0808], no game negative; expected +.06 to +.08 on our match. A wrong gate turns live possession into a stoppage and cuts a spell, the most expensive error the score charges.

<img src="figs/fig_stage_gate.gif" style="width:100%">
<div class="caption" style="font-size:0.9em; color:#555; margin:2px 0 14px 0;">Stage 3, the missing-ball gate: the ball is gone and the players are still, so the stretch becomes D. Local render.</div>

**Stage 4 — the in-gap decoder.** In a missing-ball gap, team shape says which team is wider and more likely to have the ball, so the decoder may insert a hand-over. It adds +.0052 [+.0038, +.0065], but only 3.9% of our frames are eligible, so expected +.002 to +.005. The clip shows the mechanism, not a success: an H island where the answer stays A.

<img src="figs/fig_stage_ingap.gif" style="width:100%">
<div class="caption" style="font-size:0.9em; color:#555; margin:2px 0 14px 0;">Stage 4, the in-gap decoder: team shape inserts a hand-over inside a missing-ball gap. Local render.</div>

**Stage 5 — grown stoppages.** A resting ball is a real stoppage but only a sliver. Extend each moment to the stoppage around it, with a radius and a thirty-second cap: +.0055 [.0044, .0066], expected +.001 to +.005. Only safe on top of the gate; on raw Baseline+ most changed frames are live play.

<img src="figs/fig_stage_grown.gif" style="width:100%">
<div class="caption" style="font-size:0.9em; color:#555; margin:2px 0 14px 0;">Stage 5, grown stoppages: each resting-ball moment is extended to the stoppage around it. Local render.</div>

**Stage 6 — the sub-gap rule.** Some stoppages sit inside gaps the gate left alone. A weaker rule fires when the stoppage probability stays above one half for two seconds: +.0044 [.0033, .0056], expected about +.002. On a live gap it creates a D island inside a possession.

<img src="figs/fig_stage_subgap.gif" style="width:100%">
<div class="caption" style="font-size:0.9em; color:#555; margin:2px 0 14px 0;">Stage 6, the sub-gap rule: a stoppage inside a gap the main gate left alone. Local render.</div>

The build applies them gate, grown D, in-gap, sub-gap, then the short-spell floor, because grown D must be excluded from the re-decode and the sub-gap rule must see the gate's output. The numbers are marginal, not a running sum. The floor **is** wired in the frozen build: deleting H/A spells under one and a half seconds adds +.0099 [.0063, .0138] on the pre-add-on stack and +.0113 on the frozen labels, and it is the last label write on every return path. It is a trade — real short possessions are deleted to remove more false ones — not free pruning.

| stage | what it does | practice spellF1 | marginal gain | expected on our match |
|---|---|---|---|---|
| smooth50 | nearest player, majority-smoothed | .3037 | — | — |
| Baseline+ | hysteresis, merge, 2.5 m radius | .3259 | +.0222 [+.0058, +.0403] | level to slightly ahead |
| gate | missing ball + still players -> D | .3987 | +.0728 [+.0649, +.0808] | +.06 to +.08 |
| in-gap decoder | team shape inside a missing-ball gap | .4039 | +.0052 [+.0038, +.0065] | +.002 to +.005 |
| grown stoppages | resting ball -> whole stoppage | — | +.0055 [.0044, .0066] | +.001 to +.005 |
| sub-gap rule | second gate inside unfired gaps | — | +.0044 [.0033, .0056] | +.002 |

## 5. The two clues that carry it

At the top of the notebook the ball is gone for six seconds. Two instruments were asked whether that time is a stoppage: a classifier trained on player motion, and the broadcast camera's own ball track, which follows the ball through gaps the tactical tracker misses. Neither uses the answers.

<img src="figs/fig_gate_instruments.png" style="width:100%">
<div class="caption" style="font-size:0.9em; color:#555; margin:2px 0 14px 0;">Two independent instruments agree on how much of the missing time is dead. Aggregates only.</div>

The player-motion classifier flags 85.6% [80.8%, 89.2%] of our missing time as dead; the broadcast route puts 84.5% of it dead, with the verified-live bucket at 2.1%–6.9%, carrying the dead share to about 82%–87%. After correcting for what each cannot see, the probability that a missing frame is dead is .93–.99. Two instruments with different failure modes agreeing is the strongest label-free evidence we have — not a license to call every gap dead, so the gate also requires still players.

The second clue is **team shape**. The team with the ball spreads out; the defender stays compact. On live practice frames the possession team's width is 13.9 m against 11.3 m, a difference of about 2.5 m that separates them with AUC about .88–.90. The catch is where it applies: inside a dead gap the same difference collapses to −0.4 m, because both teams walk into position. So it works only where the gap is live, and only 3.9% of our frames are eligible against 17% to 44% on the practice set — hence +.002 to +.005, not the large number the decoder scored alone.

<img src="figs/fig_width_schematic.png" style="width:100%">
<div class="caption" style="font-size:0.9em; color:#555; margin:2px 0 14px 0;">The team with the ball is wider; the defending team is narrow. Synthetic.</div>

<img src="figs/fig_shape_deform.gif" style="width:100%">
<div class="caption" style="font-size:0.9em; color:#555; margin:2px 0 14px 0;">The shape clue in a real missing-ball gap: the wide team changes as the owner changes. Local render.</div>

## 6. What we tried and dropped

The most tempting idea was to find the kick. On clean public data the textbook detector finds 75%–83% of them; on our ball it fails, because the path is smoothed over about a third of a second. Smoothing the practice ball the same way reproduces the failure.

That set the pattern: a clue that works on clean data is not evidence until it survives our noise. The table carries the number that killed each idea. The duration model was the expensive attempt — an explicit-duration HSMM scores −.0090 [−.0163, −.0024], negative in nine of ten games, while a sticky HMM without the prior scores +.0084 [−.0025, +.0180] — positive but including zero, so tier-2. The prior is what loses. Restart-team-by-type changed accuracy by −0.4 points; splitting long blocks scores −.0011; crowd loudness led by +.027, below our five-point floor; a public audio model heard a synthetic whistle at .115 but the real one at .001.

| idea | what killed it |
|---|---|
| kicks from ball kinematics | only 75%-83% recall on clean data; our ball is smoothed, so it fails here too |
| ball-player coupling | cost −.002 to −.011 spellF1 when added to the decoder |
| video ball detection | 36% recall on frames where tracking had the ball, 82% false positives on frames where it did not; the ball is 8 px to 10 px |
| explicit-duration HSMM | −.0090 [−.0163, −.0024], negative in nine of ten games |
| restart team by stoppage type | −0.4 points restart accuracy and negative spellF1 in every variant |
| split long stoppage blocks | −.0011, negative in ten of ten games |
| crowd loudness | lead +.027, below the five-point floor |
| public audio-event model | synthetic whistle .115, this match's whistle .001 |

<img src="figs/fig_killed.png" style="width:100%">
<div class="caption" style="font-size:0.9em; color:#555; margin:2px 0 14px 0;">Each dropped arm's decisive spellF1 delta against the shipped stack. Synthetic from numbers only.</div>

## 7. Results on the frozen build

The configuration is frozen (D-029). This is the dev reference: ten practice games, six primary cells, game-cluster intervals. The frozen row is **one run of the whole stack**, not a sum of the component deltas above.

| row | dev-v1 spellF1 | marginal gain | reading |
|---|---|---|---|
| smooth50 | .3037 | — | reference floor |
| Baseline+ | .3259 | +.0222 [+.0058, +.0403] vs smooth50 | about level to slightly ahead |
| D gate | .3987 | +.0728 [+.0649, +.0808] vs Baseline+ | the D sensor |
| ship (gate + in-gap) | .4039 | +.0052 [+.0038, +.0065] vs gate | D-021 reference |
| **frozen master** | **.4247** | **+.0208 [+.0169, +.0250] vs ship** | **0/10 games negative** |

| comparison | delta | 95% game-cluster CI | games negative |
|---|---|---|---|
| master − ship | **+.0208** | [+.0169, +.0250] | 0/10 |
| master − Baseline+ | +.0988 | [+.0909, +.1068] | 0/10 |

Against the organizers' floor, the pre-floor stack is +.1003 [+.0877, +.1130]; the frozen stack and that floor were never measured as a pair, so we do not quote the arithmetic difference.

The gain is smaller on our match. Reach-scaled, the frozen stack should beat the pre-floor stack by +.011 to +.016, not the +.0208 [+.0169, +.0250] measured on dev, because dev is about .06 easier and the new levers reach less of our match. Everything that moved moved for one reason: the D layer. The gate supplies most of the +.0988 [+.0909, +.1068] over Baseline+; the grown, sub-gap and floor rules add a few thousandths each.

What the file itself looks like, counted row by row from the frozen CSV:

| diagnostic | frozen CSV |
|---|---|
| rows / graded frames | 199,319 / 158,857 |
| H / A / D / X | 50,478 / 35,720 / 72,659 / 40,462 |
| D share of graded | .4574 |
| spells (maximal H/A runs) | 370 |
| D stretches | 138 (111 at least 1.5 s) |
| ball in play | 57.47 min of 105.9 min graded |
| sha256 | `ee7b5b6b…46d8` |

370 spells, 57.47 min of play and .4574 dead are in the right football neighbourhood; they do **not** say the labels are right. Our match is dead-heavier than the football prior because of the cooling break and tracker-lost visible balls. These are label-free descriptions, not scores.

The public window, one run, reported and never a selector:

| run | spellF1 | macroF1 | predicted / GT spells |
|---|---|---|---|
| **ours (frozen)** | .4686 | .7199 | 18/17 |
| smooth125 | .4242 | .5812 | 16/17 |
| smooth50 (published floor) | .3256 | .6207 | 26/17 |
| smooth25 | .2868 | .6200 | 36/17 |
| raw | .0378 | .5972 | 131/17 |

Sorted by spellF1 we are first on both metrics, but seventeen true possessions is inside noise and the prediction has eighteen. Nothing was chosen here. There is no defensible absolute full-match score: every interval above is a paired delta, dev is easier than the match, and the public window is a direction check.

## 8. Limits, and what would fix them

The practice set's eight blind spots (D-022) bound what transfers:

| blind spot | what it does to the numbers |
|---|---|
| no cooling breaks or VAR stops in dev | part of the sim-to-real gap is structural; our D share is legitimately higher |
| Metrica over-calls D around advantage play | 3 of 10 games; 20× spread in its own present-D share |
| sliver gaps | ~10 gaps of 2–4 s per game, 0.5% of graded; in-gap and sub-gap gains are upper-ish |
| resting-ball recall on dev | .19 / .085 against .45 clean; grown D is measured pessimistically |
| gap-edge spread | symmetric, sd ~1 s; the fitted spread is 2×–4× larger |
| geometry (checked, closed) | nearest player 2.29 m vs 2.35 m; proximity cues are not inflated |
| ten games, three seeds, level 1 | CIs omit degrader misspecification; one match |
| level, not ranking | .3037 on dev against .243 published; only orderings transfer |

The biggest weakness is recall, not D. On the practice set 79% of real possessions are missed — about one in five is caught. That is a dev figure from the gate-plus-in-gap stack, not a match measurement, which needs labels we do not have. The owner-change family is most of the loss, and much of it has no tracked ball.

Hand-over scatter is **evidence-limited, not rule-limited** (E83). 46.9% [43.9%, 49.5%] of matched hand-overs are over a second off (31.2% early, 15.7% late), and 93.4% of that tail is the 2.5 m nearest-player cue itself: a duel has no timing information, and on a pass the receiver is nearest long before the touch. Only 0.69% of tail flips are moved by half a second or more by any stage after Baseline+, so no post-processing rule reaches the +.0796 oracle. The route is a contact detector at about 0.16 s; on this tracking the ball is smoothed over several frames, which puts it out of reach.

What a different data source would fix: contact-level tracking would attack the tail and reopen the tier-2 `anchor` lever; a practice regime with cooling breaks and video-review stops would test the learned arms where they gain, since their advantage is confined to dead-frame motion the ten games lack; a GPU ball detector would add ball coverage but not contact timing, and nothing may depend on it (D-004). We would not retry touch rebuilds, explicit-duration priors, a learned decoder instead of the gate, splitting long D blocks, or restart-team-from-type: each has a decisive negative above.

## What changed since the freeze

The freeze itself did not move: the submission is still the same file, and the headline result is still +.0208 [+.0169, +.0250] against the pre-floor stack. What changed is how much of it we are willing to claim. Post-freeze studies re-measured the small components, asked whether the practice ranking survives a different draw or a different severity level, took the two large orderings to twenty held-out SkillCorner games, and corrected the ball-in-play denominator. None of them reopened the frozen file; all of them changed what a reader should believe about the numbers in it.

The in-gap decoder is the clearest example. It was credited +.0052 [+.0038, +.0065] when it landed on the gate-only stack; on the current stack its leave-one-out marginal is +.0017 [+.0004, +.0030]. Those are different objects: most of the fall is work that grown D and the short-spell floor now do first, not a reversal of the decoder. The small number is also not sign-established. Any dev delta below .004 is quoted as *level with zero*, and this one is well below it: six degradation draws span −.0009 to +.0031, and the two-level interval that resamples draws as well as games is [−.0011, +.0041], which includes zero. We keep the stage — removing it is not justified either — and stop calling it a gain.

The big dev result is not a lucky draw or a lucky severity. Fresh degradation draws give +.0203 [+.0166, +.0241] over six draws, with the frozen three-seed number inside that interval. Scaling the practice data to 0.8× and 1.2× of its fitted severity leaves the same order, and the smallest adjacent margin is +.0195 at the cleaner level. These are ranking checks, not a second practice set, and no level was re-selected.

On twenty held-out SkillCorner games the two large orderings keep their sign under all four derived-D rules: the frozen stack beats Baseline+ by +.0361 to +.0567, and the pre-floor stack beats it by +.0215 to +.0407. The third ordering, the frozen stack against the pre-floor stack, is positive but only +.0092 to +.0166, below the 0.02 bar we require for a held-out claim. The held-out set therefore confirms the two large gains and cannot resolve the small one; the dev grid still decides it.

The ball-in-play check was a denominator error, now fixed. The frozen file has 57.47 min of live play on a 103.95 min playing clock, a live share of 55.28%; the ten external games average 58.21% ± 6.38%. On the rate axis that is −0.46 standard deviations from the external mean — inside the noise, not the shortfall a raw-minute comparison suggested. The raw-minute axis is confounded by our unusually long clock and is no longer quoted.

The stage attribution is a substitution result, not a sum. Leave-one-out at one seed puts the floor at +.0114, sub-gap at +.0062, grown D at +.0045 and in-gap at −.0009; the three post-freeze stages sum to about +.0221 against the measured +.0208 [+.0169, +.0250], an interaction of −.0013. Most of that interaction is grown D handing the floor work it used to do. Read the table as what each stage is worth given the other four, not as a budget that adds up.

## Code, data and acknowledgements

The build is one command in `solution/`: `python -m possession.make_submission`. The pipeline modules are `baseline_plus.py`, `dgate.py`, `growd.py`, `ingap.py`, `subgap.py` and `postprocess.py`; the harness is `dev_v1.py`; the renderer is `render.py`. The notebook is built by `scripts/walkthrough_numbers.py` and `scripts/build_walkthrough.py` from `numbers.json`.

The practice data is public tracking from DFL, Metrica and SkillCorner, used under their licenses and never redistributed. The match data is the organizer's and stays on this machine. The scoring code is the organizer's `evaluation/possession_metrics.py`, imported, not re-implemented. Figures that render tracking are local-only under D-002; the public profile replaces them with placeholders.

## Appendix: traceability

Every measured number in the prose is a key in `notebooks/numbers.json`, with its value, source, exact command and whether `--check` recomputes it from the live pipeline. The table lists the headline keys; the rest are in that file, and the figure table names the eighteen figures.

| key | value | section | recomputed |
|---|---|---|---|
| master_dev | .4247 | 7 | no |
| master_over_ship | +.0208 [+.0169, +.0250] | 7, post | no |
| master_over_baseline_plus | +.0988 [+.0909, +.1068] | 7 | no |
| master_neg | 0/10 | 7 | no |
| expected_master | +.011 to +.016 | 7 | no |
| smooth50_dev | .3037 | 3, 4 | yes |
| baseline_plus_dev | .3259 | 4 | yes |
| gate_dev | .3987 | 4 | yes |
| ship_dev | .4039 | 4 | yes |
| ship_over_smooth50 | +.1003 [+.0877, +.1130] | 7 | yes |
| gate_over_baseline_plus | +.0728 [+.0649, +.0808] | 4 | yes |
| ingap_over_gate | +.0052 [+.0038, +.0065] | 4, post | yes |
| grown_delta | +.0055 | 4 | no |
| subgap_delta | +.0044 | 4 | no |
| postprocess_delta | +.0099 | 4 | no |
| postprocess_delta_current | +.0113 | 4 | no |
| pred_spells | 370 | 7 | yes |
| d_stretches | 138 | 7 | yes |
| pred_d_share_graded | .4574 | 7 | yes |
| ball_in_play_min | 57.47 min | 7, post | yes |
| graded_min | 105.9 min | 7 | yes |
| csv_sha8 | ee7b5b6b…46d8 | 4 | yes |
| public_ship_spellf1 | .4686 | 7 | no |
| public_ship_macro | .7199 | 7 | no |
| public_ship_spells | 18 | 7 | no |
| missed_share | 79% | 8 | no |
| handover_tail | 46.9% | 8 | no |
| handover_cue_share | 93.4% | 8 | no |
| stage_moved | 0.69% | 8 | no |
| handover_oracle | +.0796 | 8 | no |
| e25_dead_share | 85.6% | 5 | no |
| e36_dead_share | 84.5% | 5 | no |
| shape_width_diff_m | 2.5 m | 5 | yes |
| shape_dead_gap_m | −0.4 m | 5 | yes |

| figure | section | profile |
|---|---|---|
| fig_gate_instruments | 5 | both |
| fig_killed | 6 | both |
| fig_match_gap | 1 | local only |
| fig_missingness | 3 | both |
| fig_nearest_player | 1 | both |
| fig_practice_clean | 3 | local only |
| fig_practice_degraded | 3 | local only |
| fig_shape_deform | 5 | local only |
| fig_spellf1 | 2 | both |
| fig_spellf1_curve | 2 | both |
| fig_stage_baseline_plus | 4 | local only |
| fig_stage_gate | 4 | local only |
| fig_stage_grown | 4 | local only |
| fig_stage_ingap | 4 | local only |
| fig_stage_smooth50 | 4 | local only |
| fig_stage_subgap | 4 | local only |
| fig_three_labels | 1 | both |
| fig_width_schematic | 5 | both |